In [8]:
%run data_loader.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.ensemble import HistGradientBoostingClassifier

Config loaded
preprocess_binary() defined
load_all_datasets() defined
Project root:          /Users/yogeshpatel/Desktop/Tilak/IDS_project
CICIDS2018 folder exists: False
load_cicids2018() defined


In [9]:
datasets = load_all_datasets()

print(f"Loaded: {list(datasets.keys())}")

Loading 8 dataset(s): ['monday', 'bruteforce', 'dos', 'web_attacks', 'infiltration', 'botnet', 'portscan', 'ddos']

  [monday]  Monday-WorkingHours.pcap_ISCX.csv
           raw (529918, 79)  ->  clean (502650, 81)  (attack rate 0.0%)
  [bruteforce]  Tuesday-WorkingHours.pcap_ISCX.csv
           raw (445909, 79)  ->  clean (421626, 81)  (attack rate 2.2%)
  [dos]  Wednesday-workingHours.pcap_ISCX.csv
           raw (692703, 79)  ->  clean (610492, 81)  (attack rate 31.7%)
  [web_attacks]  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
           raw (170366, 79)  ->  clean (164179, 81)  (attack rate 1.3%)
  [infiltration]  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
           raw (288602, 79)  ->  clean (252790, 81)  (attack rate 0.0%)
  [botnet]  Friday-WorkingHours-Morning.pcap_ISCX.csv
           raw (191033, 79)  ->  clean (184044, 81)  (attack rate 1.1%)
  [portscan]  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
           raw (286467, 79)  ->  clea

In [10]:
combined = pd.concat(datasets.values(), ignore_index=True)

print("Combined shape:", combined.shape)

print("\nBinary label distribution:")
print(combined["Label_Binary"].value_counts())

Combined shape: (2572640, 81)

Binary label distribution:
Label_Binary
0    2146899
1     425741
Name: count, dtype: int64


In [11]:
X = combined.drop(columns=["Label", "Label_Binary", "Source_File"])
y = combined["Label_Binary"]

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (2058112, 78)
X_test : (514528, 78)


In [13]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [14]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_depth=8,
    random_state=42
)

hgb_model.fit(X_train, y_train)

HistGradientBoostingClassifier(max_depth=8, random_state=42)

In [15]:
y_pred_hgb = hgb_model.predict(X_test)
y_scores_hgb = hgb_model.predict_proba(X_test)[:, 1]

In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("HistGradientBoosting Results")

print("Accuracy :", round(accuracy_score(y_test, y_pred_hgb), 4))
print("Precision:", round(precision_score(y_test, y_pred_hgb), 4))
print("Recall   :", round(recall_score(y_test, y_pred_hgb), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_hgb), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_scores_hgb), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))

HistGradientBoosting Results
Accuracy : 0.9989
Precision: 0.9964
Recall   : 0.997
F1 Score : 0.9967
ROC-AUC  : 1.0

Confusion Matrix:
[[429076    304]
 [   252  84896]]
